# Notebook 01 — Dataset Overview

This notebook explores the NovaSaaS knowledge base corpus and evaluation set:
- Document statistics and category distribution
- Chunk size distribution and coverage analysis
- Eval set composition and difficulty balance
- Question-answer pair quality review

**No API keys required.** All data is loaded from local files.


In [ ]:
import sys
from pathlib import Path

# Add src to path for notebook use
sys.path.insert(0, str(Path().absolute().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

from llm_evals_lab.config import load_config
from llm_evals_lab.data.loader import CorpusLoader
from llm_evals_lab.data.evalset import EvalSetLoader

cfg = load_config()
print('Config loaded. Project root:', cfg.project_root)

## 1. Source Document Corpus

In [ ]:
loader = CorpusLoader(cfg.raw_dir(), cfg.processed_dir())
docs = loader.load_documents()
chunks = loader.load_chunks()

print(f'Documents: {len(docs)}')
print(f'Chunks: {len(chunks)}')
print()

doc_df = pd.DataFrame([
    {
        'doc_id': d.doc_id,
        'title': d.title,
        'category': d.category,
        'source_type': d.source_type,
        'word_count': d.word_count,
        'chunk_count': sum(1 for c in chunks if c.doc_id == d.doc_id)
    }
    for d in docs
])

doc_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Word count per document
axes[0].barh(doc_df['doc_id'], doc_df['word_count'], color='#4C72B0')
axes[0].set_xlabel('Word Count')
axes[0].set_title('Words per Document')

# Category distribution
cat_counts = doc_df['category'].value_counts()
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Category Distribution')

# Chunks per doc
axes[2].bar(doc_df['doc_id'], doc_df['chunk_count'], color='#DD8452')
axes[2].set_xlabel('Document ID')
axes[2].set_title('Chunks per Document')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/01_corpus_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 2. Chunk Statistics

In [ ]:
chunk_df = pd.DataFrame([
    {
        'chunk_id': c.chunk_id,
        'doc_id': c.doc_id,
        'word_count': c.word_count,
        'chunk_order': c.chunk_order,
        'category': c.metadata.get('category', 'unknown')
    }
    for c in chunks
])

print('Chunk word count statistics:')
print(chunk_df['word_count'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(chunk_df['word_count'], bins=20, color='#4C72B0', edgecolor='white')
ax.axvline(chunk_df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {chunk_df["word_count"].mean():.0f}')
ax.set_xlabel('Words per Chunk')
ax.set_ylabel('Count')
ax.set_title('Chunk Size Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Evaluation Set

In [ ]:
eval_loader = EvalSetLoader(cfg.eval_dir())
examples = eval_loader.load()

eval_df = pd.DataFrame([
    {
        'example_id': e.example_id,
        'category': e.category.value,
        'difficulty': e.difficulty.value,
        'is_answerable': e.is_answerable,
        'question_words': len(e.question.split()),
        'reference_words': len(e.reference_answer.split()),
        'key_points': len(e.expected_key_points),
        'expected_docs': len(e.expected_doc_ids),
    }
    for e in examples
])

print(f'Total examples: {len(examples)}')
print(f'Answerable: {eval_df.is_answerable.sum()}')
print(f'Unanswerable: {(~eval_df.is_answerable).sum()}')
eval_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cat_counts = eval_df['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color='#55A868')
axes[0].set_title('Examples by Category')
axes[0].set_xlabel('Category')
axes[0].tick_params(axis='x', rotation=30)

diff_counts = eval_df['difficulty'].value_counts()
colors = {'easy': '#55A868', 'medium': '#F5A623', 'hard': '#E24A33'}
bar_colors = [colors.get(d, 'gray') for d in diff_counts.index]
axes[1].bar(diff_counts.index, diff_counts.values, color=bar_colors)
axes[1].set_title('Examples by Difficulty')
axes[1].set_xlabel('Difficulty')

plt.tight_layout()
plt.show()

In [ ]:
# Print sample examples
for ex in examples[:3]:
    print(f'[{ex.example_id}] [{ex.difficulty.value}] [{ex.category.value}]')
    print(f'  Q: {ex.question}')
    print(f'  A: {ex.reference_answer[:120]}...')
    print(f'  Expected docs: {ex.expected_doc_ids}')
    print()